### Установка библиотек

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

from datasets import load_dataset
import random
import numpy as np
import json

from transformers import BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_cosine_schedule_with_warmup

In [2]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

### Загрузка датасета для анализа тональности

In [3]:
ds = load_dataset("ai-forever/kinopoisk-sentiment-classification")

Using the latest cached version of the dataset since ai-forever/kinopoisk-sentiment-classification couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/jupyter/datasphere/project/datasetscache/ai-forever___kinopoisk-sentiment-classification/default/0.0.0/4937df51b02a4c748b38bace5d749524fd90ae4a (last modified on Sat Jan 24 11:20:52 2026).


### Токенизатор (и модель DeepPavlov)

In [4]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

### Реализация срезов

In [5]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                sentences[i],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        encoding['labels'] = [labels[i]]
        result.append({key : value[0] for key, value in encoding.items()})

    return result

### Кастомный датасет

In [6]:
class SemDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):

        if isinstance(idx, slice):
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]

            encoding = self.tokenizer(
                tokens,
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            encoding['labels'] = [tag]
            return {key : value[0] for key, value in encoding.items()}

### Параметры

In [7]:
batch_size = 16
max_length = 512
epochs = 3
learning_rate = 2e-5
weight_decay = 0.01

### Создание датасетов для тренировки, валидации и тестирования

In [8]:
dataset_train = SemDataset(ds['train']['text'], ds['train']['label'], tokenizer, max_length)
dataset_test = SemDataset(ds['test']['text'], ds['test']['label'], tokenizer, max_length)
dataset_val = SemDataset(ds['validation']['text'], ds['validation']['label'], tokenizer, max_length)

### Даталоадеры

In [9]:
test_loader = DataLoader(dataset_test, batch_size, pin_memory=True)
train_loader = DataLoader(dataset_train, batch_size)
val_loader = DataLoader(dataset_val, batch_size)

### Предобученная модель DeepPavlov/rubert-base-cased

In [10]:
num_labels = len(set(ds['train']['label']))
model = BertForSequenceClassification.from_pretrained('DeepPavlov/rubert-base-cased', num_labels= num_labels)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 383.21it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.

### Функция для подсчета метрик

In [11]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

### Обучение с оптимизатором и шедулером

In [12]:
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()))
total_steps = len(dataset_train) * epochs * batch_size

scheduler = get_cosine_schedule_with_warmup(
  optimizer,
  num_warmup_steps=total_steps*0.05,
  num_training_steps=total_steps
)

### Задаем директории для сохранения чекпоинтов, конфигураций и метрик

In [ ]:
import os
from pathlib import Path
outputs_dir = os.makedirs("../outputs", exist_ok=True)
model_dir = os.makedirs("../model_tokenizer", exist_ok=True)
model_dir = Path("../model_tokenizer")
outputs_dir = Path("../outputs")
checkpoints_dir = os.path.join(model_dir, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)
checkpoints_dir = Path("../model_tokenizer/checkpoints")

### Агрументы

In [ ]:
training_args = TrainingArguments(
    output_dir=checkpoints_dir,
    eval_strategy="epoch",
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="epoch"
)

### Дефолтный трейнер

In [39]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, scheduler)
    )

### Тренировка и валидация, сохранение конфигураций модели и токенайзера

In [40]:
train_metrics = trainer.train().metrics

with open(os.path.join(outputs_dir, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

with open(os.path.join(outputs_dir, "eval_metrics.json"), "w") as f:
    json.dump(eval_results, f, indent=2)

trainer.save_model("../model_tokenizer")
tokenizer.save_pretrained("../model_tokenizer")

 33%|███▎      | 657/1971 [20:21<31:56,  1.46s/it]

{'loss': '0.7664', 'grad_norm': '12.22', 'learning_rate': '3.806e-05', 'epoch': '1'}



100%|██████████| 94/94 [00:40<00:00,  2.54it/s]
                                                  
 33%|███▎      | 657/1971 [21:02<31:56,  1.46s/it]
                                               

Confusion Matrix:
 [[171 222 107]
 [ 18 128 354]
 [  2  19 479]]
{'eval_loss': '1.016', 'eval_accuracy': '0.5187', 'eval_f1': '0.4849', 'eval_precision': '0.5839', 'eval_recall': '0.5187', 'eval_runtime': '40.65', 'eval_samples_per_second': '36.9', 'eval_steps_per_second': '2.313', 'epoch': '1'}



 67%|██████▋   | 1314/1971 [41:38<16:01,  1.46s/it] 

...

{'train_runtime': '3826', 'train_samples_per_second': '8.233', 'train_steps_per_second': '0.515', 'train_loss': '0.6828', 'epoch': '3'}



100%|██████████| 94/94 [00:39<00:00,  2.37it/s]


Confusion Matrix:
 [[460  33   7]
 [231 129 140]
 [ 52  73 375]]
Evaluation Results: {'eval_loss': 0.8377922177314758, 'eval_accuracy': 0.6426666666666667, 'eval_f1': 0.608340135004839, 'eval_precision': 0.6288128946990477, 'eval_recall': 0.6426666666666666, 'eval_runtime': 40.2461, 'eval_samples_per_second': 37.271, 'eval_steps_per_second': 2.336, 'epoch': 3.0}


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.35s/it]


('../model_tokenizer/tokenizer_config.json',
 '../model_tokenizer/tokenizer.json')

### Тестирование модели

In [41]:
test_results = trainer.evaluate(eval_dataset=test_loader.dataset, metric_key_prefix="test")
print(f"Evaluation Results: {test_results}")

with open("../outputs/test_metrics.json", "w") as f:
    json.dump(test_results, f, indent=2)

100%|██████████| 94/94 [00:39<00:00,  2.41it/s]

Confusion Matrix:
 [[454  42   4]
 [222 123 155]
 [ 45  70 385]]
Evaluation Results: {'test_loss': 0.8476894497871399, 'test_accuracy': 0.6413333333333333, 'test_f1': 0.6052981713080235, 'test_precision': 0.6202686140558268, 'test_recall': 0.6413333333333333, 'test_runtime': 39.5663, 'test_samples_per_second': 37.911, 'test_steps_per_second': 2.376, 'epoch': 3.0}
